In [1]:
import numpy as np

Suppose you have a factor graph with variables $X,Y,Z$ and a factor $f(X,Y,Z)$.  You want to update the factor-to-variable message from $f$ to $X$ given by the formula,
\begin{equation}
  \mu_{f\rightarrow x}(x) = \sum_z \sum_y f(x,y,z) \mu_{z \rightarrow f}(z) \mu_{y \rightarrow f}(y)
\end{equation}
In our example, assume that variables can take the following values:
\begin{equation}
  X \in \{1, \ldots, 8\} \qquad Y \in \{1,2\} \qquad Z \in \{1,2,3\}
\end{equation}
We will start by defining our factor and incoming messages.  

In [2]:
mu_y2f = np.random.rand(2)
mu_z2f = np.random.rand(3)
f = np.random.rand(8,2,3)
print('mu_y2f shape: ', mu_y2f.shape)
print('mu_z2f shape: ', mu_z2f.shape)
print('Factor shape: ', f.shape)

mu_y2f shape:  (2,)
mu_z2f shape:  (3,)
Factor shape:  (8, 2, 3)


The values of the messages and factor ar arbitrary, so I just chose random values.  Importantly, the size of the factor $f(x,y,z)$ is $8 \times 2 \times 3$.  This is a multidimensionaly array whose axes align with each of the arguments.  The messages are 1D arrays of the same length as the corresponding variables.  

Now we use Numpy.tensordot to compute the product of incoming messages,
\begin{equation}
  \mu_{z \rightarrow f}(z) \mu_{y \rightarrow f}(y)
\end{equation}

In [3]:
msg_prod = np.tensordot(mu_y2f, mu_z2f,axes=0)
print('msg_prod shape: ', msg_prod.shape)

msg_prod shape:  (2, 3)


Numpy.tensordot with argument axis=0 automatically aligns dimensions of its argument.  The product is a 2D array of size $2 \times 3$.  The element tmp_msg[i,j] = mu_y2f[i] * mu_z2f[j]. This product aligns with the 2nd and 3rd dimensions of the factor.  We will now replicate this across the first dimension (the X dimension).

In [4]:
tmp_msg = np.tile(msg_prod, (8, 1, 1))
print('tmp_msg shape: ', tmp_msg.shape)

tmp_msg shape:  (8, 2, 3)


Now, tmp_msg has the same shape as the factor.  The Numpy.tile function creates 8 copies of the 2D array, where each copy is aligned to the 1st axis.  That means, each slice along the first axis is identical, which can check...

In [5]:
# Choose two slices arbitrarily
check_1 = np.squeeze(tmp_msg[3,:,:])
check_2 = np.squeeze(tmp_msg[7,:,:])
print('Check 1: ', check_1)
print('Check 2: ', check_2)
print('The same!')

Check 1:  [[0.12741917 0.13021238 0.84547282]
 [0.02054304 0.02099337 0.1363106 ]]
Check 2:  [[0.12741917 0.13021238 0.84547282]
 [0.02054304 0.02099337 0.1363106 ]]
The same!


Now that everything is the same size, we can compute the final message.  We first multiply the factor with the replicated message product element-wise:

In [6]:
tmp_msg = f * tmp_msg
print('tmp_msg shape: ', tmp_msg.shape)

tmp_msg shape:  (8, 2, 3)


Now we just sum over the 2nd and 3rd axes (corresponding to $y$ and $z$).  The result is the factor-to-variable message $\mu_{f \rightarrow x}(x)$.

In [7]:
mu_f2x = np.sum(tmp_msg, axis=(1,2))
print('Message: ', mu_f2x)
print('Shape: ', mu_f2x.shape)

Message:  [0.24521528 0.60436252 1.18748229 0.34408828 0.74763862 0.431988
 0.56450834 0.46566119]
Shape:  (8,)


Notice that the final message $\mu_{f \rightarrow x}(x)$ is an array of length 8.  This is because it is a function of the variable $X$ which takes on 8 values.  One last step is we should normalize the message, to avoid underflow later on...

In [8]:
mu_f2x = mu_f2x / np.sum(mu_f2x) # can divide by a constant
print('Message: ', mu_f2x)

Message:  [0.05341282 0.13164231 0.25865751 0.07494934 0.16285072 0.09409567
 0.12296126 0.10143037]


That is one way to compute the factor-to-variable message.  Note that we can use Numpy broadcasting to avoid explicitly using Numpy.tile().  

In [9]:
# Compute factor-to-variable message
mu_f2x_again = f * msg_prod[np.newaxis, :, :] # <== Braodcast 1st dim.
mu_f2x_again = np.sum(mu_f2x_again, axis=(1,2))
mu_f2x_again /= np.sum(mu_f2x_again) # Normalize
mu_f2x_again 

array([0.05341282, 0.13164231, 0.25865751, 0.07494934, 0.16285072,
       0.09409567, 0.12296126, 0.10143037])

Observe that we get the same result with fewer lines of code, and better memory usage.  We can write even shorter and more efficient code by using Numpy.tensordot().  The following lines compute the same message. The axes=((1,2),(0,1)) argument tells Numpy to align the 1st and 2nd axes of the factor with the 0th and 1st axes of the incoming message product (the Y and Z dimensions), then sum them out...

In [10]:
mu_f2x_again = np.tensordot(f, msg_prod, axes=((1,2),(0,1)))
mu_f2x_again /= np.sum(mu_f2x_again) # Normalize
mu_f2x_again 

array([0.05341282, 0.13164231, 0.25865751, 0.07494934, 0.16285072,
       0.09409567, 0.12296126, 0.10143037])

What if we want to send a message from factor $f$ to variable $y$?  In this case, $y$ is the 2nd dimension (1st axis) of the factor array.  We just need to keep track of which variables are on each dimension.  In this case, the formula is,
\begin{equation}
  \mu_{f \rightarrow y}(y) = \sum_x \sum_z f(x,y,z) \mu_{x\rightarrow f}(x) \mu_{z\rightarrow f}(z)
\end{equation}
We just need to make sure to keep track of dimensions and align them properly...

In [14]:
# random x-to-f message
mu_x2f = np.random.rand(8)

# compute product of incoming messages
msg_prod_2 = np.tensordot(mu_x2f, mu_z2f, axes=0)
print('msg_prod_2 shape: ', msg_prod_2.shape)

# compute message with broadcasting
tmp_msg_2 = f * msg_prod_2[:,np.newaxis,:]
mu_f2y = np.sum(tmp_msg_2, axis=(0,2)) # sum over X and Z dimenions
mu_f2y /= np.sum(mu_f2y) # normalize
print('mu_f2y: ', mu_f2y)

# compute using tensordot
mu_f2y_again = np.tensordot(f, msg_prod_2, axes=((0,2), np.arange(len(2)))) # align (0,2) axes with (0,1) axes and sum
mu_f2y_again /= np.sum(mu_f2y_again) # normalize
print('mu_f2y_again: ', mu_f2y_again)

msg_prod_2 shape:  (8, 3)
mu_f2y:  [0.52465845 0.47534155]
mu_f2y_again:  [0.52465845 0.47534155]


Observe that either method yields the same result.  Use whichever makes the most sense for you.